<a href="https://colab.research.google.com/github/Microblast56/Flyrank_ml_Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract



In [ ]:
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

print("HF_TOKEN available:", bool(hf_token))
print(
    "Token format looks correct:",
    hf_token.startswith("hf_") if hf_token else False
)

HF_TOKEN available: True
Token format looks correct: True


## 1. Unit of analysis + time window

**Lane:** Refresh / Content Opportunity Scoring.

**Source data:** `fact_content_daily_performance` from the
FlyRank internship warehouse.

**Source-table grain:** One row represents one pseudonymized client,
one content item, and one reporting date.

**Analysis grain:** For the review-ranking task, I will aggregate daily
records into one row per client-content item.

**Development window:** March 1–31, 2026. Features will use March 1–15,
and the outcome proxy will use March 16–31. This keeps the feature and
outcome periods separate.

**Output:** A ranked list of content items for human review. The ranking
is intended to support decisions, not guarantee that a refresh will work.

In [ ]:
%pip -q install duckdb pandas

import duckdb
import pandas as pd
from google.colab import userdata

# Retrieve the secret without displaying its value.
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. Check Colab Secrets and notebook access."
    )

# Connect DuckDB to the Hugging Face warehouse.
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH_PATH = (
    f"{REL}/fact_content_daily_performance/"
    "month=2026-03/*.parquet"
)

FACT_MARCH = (
    f"read_parquet('{MARCH_PATH}', hive_partitioning=true)"
)

print("DuckDB connection configured.")
print("Development partition: March 2026")
print("Feature window: March 1–15, 2026")
print("Outcome window: March 16–31, 2026")

DuckDB connection configured.
Development partition: March 2026
Feature window: March 1–15, 2026
Outcome window: March 16–31, 2026


## 2. Fields: feature / label / context / excluded

**Features:** I will use Google Search Console (GSC) measurements from March
1–15, 2026. The planned page-level features are impressions per day, clicks
per day, CTR, average search position, and the number of days with impressions.
Each feature will be calculated only from records available by March 15.

**Label / proxy:** `decline_proxy` will equal 1 when a content item's average
daily impressions during March 16–31 are less than 80% of its average daily
impressions during March 1–15; otherwise it will equal 0. I will only define
this proxy for items with positive impressions in the feature window. It is a
defined outcome proxy, not proof that a page will continue declining.

**Context:** `report_date`, `client_hash_id`, and `content_hash_id` describe
when and to which client and content item each daily observation belongs.
They will support filtering, grouping, and joining, but will not be model
features.

**Excluded:** March 16–31 measurements will be excluded from the feature
matrix because they belong to the outcome window. Client and content IDs will
not be predictive features. I will also avoid using GA4 measurements unless
their availability is explicitly verified with `ga4_data_available IS TRUE`.

**Output:** A page-level review queue that helps a content editor decide which
pages deserve further investigation. It will support prioritization, not
guarantee that a refresh will improve performance.

In [ ]:
# Keep the contract's field categories explicit and inspectable.

feature_fields = [
    "impressions_per_day_early",
    "clicks_per_day_early",
    "ctr_early",
    "avg_position_early",
    "active_days_early",
]

label_proxy = "decline_proxy"

context_fields = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
]

excluded_fields = {
    "March 16–31 outcome measurements":
        "These belong to the outcome window, not the feature window.",
    "client_hash_id and content_hash_id as model features":
        "IDs are for grouping and joins, not predictive signals.",
    "GA4 metrics without verified availability":
        "Missing or unavailable measurements must not be treated as zero."
}

print("Planned feature count:", len(feature_fields))
print("Planned features:", feature_fields)
print("Label/proxy:", label_proxy)
print("Context fields:", context_fields)
print("Excluded fields and reasons:")
for field, reason in excluded_fields.items():
    print(f"- {field}: {reason}")

Planned feature count: 5
Planned features: ['impressions_per_day_early', 'clicks_per_day_early', 'ctr_early', 'avg_position_early', 'active_days_early']
Label/proxy: decline_proxy
Context fields: ['report_date', 'client_hash_id', 'content_hash_id']
Excluded fields and reasons:
- March 16–31 outcome measurements: These belong to the outcome window, not the feature window.
- client_hash_id and content_hash_id as model features: IDs are for grouping and joins, not predictive signals.
- GA4 metrics without verified availability: Missing or unavailable measurements must not be treated as zero.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# Verification query 1: check the declared grain.
# One row should represent client + content item + reporting date.

grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS rows_at_same_grain
    FROM {FACT_MARCH}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date <  DATE '2026-04-01'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("QUERY 1 — Duplicate grain keys")
if grain_check.empty:
    print("No duplicate grain keys found in the checked March partition.")
else:
    display(grain_check)


# Verification query 2: count rows and verify the date span.

count_span = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_report_date,
        MAX(report_date) AS last_report_date
    FROM {FACT_MARCH}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date <  DATE '2026-04-01'
""").df()

print("\nQUERY 2 — March row count and date span")
display(count_span)


# Query 3: verify GSC availability for the data used by our features.
availability = con.sql(f"""
    SELECT
        COUNT(*) AS march_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS rows_with_gsc_available,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS rows_with_ga4_available
    FROM {FACT_MARCH}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

print("QUERY 3 — Availability checks")
display(availability)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

QUERY 1 — Duplicate grain keys
No duplicate grain keys found in the checked March partition.

QUERY 2 — March row count and date span


,row_count,first_report_date,last_report_date
0,9841378,2026-03-01,2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

QUERY 3 — Availability checks


,march_rows,rows_with_gsc_available,rows_with_ga4_available
0,9841378,3611061,413966


### Five-feature frame: what is knowable at decision time?

The decision moment is the end of March 15, 2026. Every feature below uses
only GSC observations from March 1–15.

- `impressions_per_day_early`: average impressions per observed reporting day; knowable because it uses only the completed feature window.
- `clicks_per_day_early`: average clicks per observed reporting day; knowable because it uses only the completed feature window.
- `ctr_early`: clicks divided by impressions in the early window; knowable because both measurements come from the feature window.
- `avg_position_early`: average available search position in the early window; knowable by the decision moment. Zero position values are treated as missing, not as rank zero.
- `active_days_early`: number of early-window days with positive impressions; knowable because those dates have already occurred.

The proxy `decline_proxy` is calculated from March 16–31, after the
decision moment. It is the outcome for this exercise, never a feature.

In [ ]:
# Build five early-window features and a separate later-window proxy.
# Source data stays in the Hugging Face warehouse.

feature_frame = con.sql(f"""
    WITH page_rollup AS (
        SELECT
            client_hash_id,
            content_hash_id,

            COUNT(DISTINCT CASE
                WHEN report_date >= DATE '2026-03-01'
                 AND report_date <  DATE '2026-03-16'
                THEN report_date
            END) AS early_observed_days,

            COUNT(DISTINCT CASE
                WHEN report_date >= DATE '2026-03-16'
                 AND report_date <  DATE '2026-04-01'
                THEN report_date
            END) AS late_observed_days,

            SUM(CASE
                WHEN report_date >= DATE '2026-03-01'
                 AND report_date <  DATE '2026-03-16'
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END) AS impressions_early,

            SUM(CASE
                WHEN report_date >= DATE '2026-03-01'
                 AND report_date <  DATE '2026-03-16'
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END) AS clicks_early,

            AVG(CASE
                WHEN report_date >= DATE '2026-03-01'
                 AND report_date <  DATE '2026-03-16'
                THEN NULLIF(gsc_avg_position, 0)
            END) AS avg_position_early,

            COUNT(DISTINCT CASE
                WHEN report_date >= DATE '2026-03-01'
                 AND report_date <  DATE '2026-03-16'
                 AND COALESCE(gsc_impressions, 0) > 0
                THEN report_date
            END) AS active_days_early,

            SUM(CASE
                WHEN report_date >= DATE '2026-03-16'
                 AND report_date <  DATE '2026-04-01'
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END) AS impressions_late

                FROM {FACT_MARCH}
        WHERE report_date >= DATE '2026-03-01'
          AND report_date <  DATE '2026-04-01'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )

    SELECT
        client_hash_id,
        content_hash_id,

        impressions_early / NULLIF(early_observed_days, 0)
            AS impressions_per_day_early,

        clicks_early / NULLIF(early_observed_days, 0)
            AS clicks_per_day_early,

        100.0 * clicks_early / NULLIF(impressions_early, 0)
            AS ctr_early,

        avg_position_early,
        active_days_early,

        CASE
            WHEN impressions_late / NULLIF(late_observed_days, 0)
                 < 0.8 * (
                     impressions_early / NULLIF(early_observed_days, 0)
                 )
            THEN 1
            ELSE 0
        END AS decline_proxy

    FROM page_rollup
    WHERE impressions_early > 0
      AND early_observed_days >= 10
      AND late_observed_days >= 10
""").df()

FEATURE_COLUMNS = [
    "impressions_per_day_early",
    "clicks_per_day_early",
    "ctr_early",
    "avg_position_early",
    "active_days_early",
]

print(f"Client-content rows in feature frame: {len(feature_frame):,}")
print(f"Feature count: {len(FEATURE_COLUMNS)}")
print("Proxy-positive rate:",
      f"{feature_frame['decline_proxy'].mean():.1%}")
print("Feature columns:", FEATURE_COLUMNS)
display(feature_frame.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Client-content rows in feature frame: 96,676
Feature count: 5
Proxy-positive rate: 31.2%
Feature columns: ['impressions_per_day_early', 'clicks_per_day_early', 'ctr_early', 'avg_position_early', 'active_days_early']


,client_hash_id,content_hash_id,impressions_per_day_early,clicks_per_day_early,ctr_early,avg_position_early,active_days_early,decline_proxy
0,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,331.600000,0.600000,0.180941,8.155725,15,0
1,client_62f4a7e64f5e0096,content_e689bc511192751a,2.333333,0.000000,0.000000,5.974359,15,0
2,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,26.533333,0.066667,0.251256,5.000423,15,1
3,client_62f4a7e64f5e0096,content_df22bda1218f13ff,44.666667,0.000000,0.000000,3.007328,15,0
4,client_62f4a7e64f5e0096,content_aa184c1b4ea518e5,8.866667,0.000000,0.000000,10.836170,15,0
5,client_62f4a7e64f5e0096,content_b4de71c8ef5c4791,125.133333,1.133333,0.905701,2.801626,15,0
6,client_62f4a7e64f5e0096,content_d7568011c4325a33,49.933333,0.066667,0.133511,5.266045,15,0
7,client_62f4a7e64f5e0096,content_e847a4dcc8af3742,64.600000,0.000000,0.000000,6.336464,15,0
8,client_62f4a7e64f5e0096,content_85b1be9944e4e19d,65.333333,0.000000,0.000000,6.634372,15,1
9,client_62f4a7e64f5e0096,content_b6a42d76effd1906,30.800000,0.066667,0.216450,19.965283,15,1


### The leakage trap: a feature that already knows the answer

For this demonstration, I will deliberately create a temporary score directly
from `decline_proxy`. Because this column copies the target, it would not be
available at the decision moment. It is an intentionally invalid feature.

I will compare Precision@50 from a simple baseline that ranks by early-window
impressions against the score using the leaked target copy. The leaked score
is expected to look perfect because it already contains the answer. This is
not a legitimate model result.

After the demonstration, I will remove the temporary leaked column and keep
the baseline Precision@50 as the honest benchmark for further work.

In [ ]:
# Deliberate leakage demonstration.
# Keep the original feature_frame unchanged.

import numpy as np

def precision_at_k(scores, labels, k=50):
    scores = np.asarray(scores)
    labels = np.asarray(labels)
    order = np.argsort(-scores)
    return float(labels[order[:k]].mean())

LABEL_COLUMN = "decline_proxy"

# Honest simple baseline: rank by early-window impressions.
y = feature_frame[LABEL_COLUMN].astype(int).to_numpy()
honest_scores = (
    feature_frame["impressions_per_day_early"].fillna(0).to_numpy()
)

honest_precision_50 = precision_at_k(honest_scores, y, k=50)

# TEMPORARY, INVALID LEAK:
# This column is an exact copy of the label.
leak_demo = feature_frame[FEATURE_COLUMNS + [LABEL_COLUMN]].copy()
leak_demo["leaked_score"] = leak_demo[LABEL_COLUMN]

leaked_precision_50 = precision_at_k(
    leak_demo["leaked_score"],
    leak_demo[LABEL_COLUMN],
    k=50
)

print(f"Honest baseline Precision@50: {honest_precision_50:.3f}")
print(f"LEAKED Precision@50 (invalid): {leaked_precision_50:.3f}")
print(
    "Leaked score is an exact copy of the target:",
    leak_demo["leaked_score"].equals(leak_demo[LABEL_COLUMN])
)

# Remove the temporary leaked column.
leak_demo.drop(columns=["leaked_score"], inplace=True)

# Confirm the kept feature list contains no label or leaked score.
assert LABEL_COLUMN not in FEATURE_COLUMNS
assert "leaked_score" not in FEATURE_COLUMNS
assert "leaked_score" not in feature_frame.columns

print("\nTemporary leaked column removed.")
print("Final feature count:", len(FEATURE_COLUMNS))
print("Final feature columns:", FEATURE_COLUMNS)
print(f"Honest benchmark retained: {honest_precision_50:.3f}")

Honest baseline Precision@50: 0.420
LEAKED Precision@50 (invalid): 1.000
Leaked score is an exact copy of the target: True

Temporary leaked column removed.
Final feature count: 5
Final feature columns: ['impressions_per_day_early', 'clicks_per_day_early', 'ctr_early', 'avg_position_early', 'active_days_early']
Honest benchmark retained: 0.420


## 4. Data limits

This analysis uses only the March 2026 partition and aggregates daily
observations into client-content rows. It does not establish whether the
results generalize to other months or unseen clients.

The warehouse has an unbalanced panel: different clients have different
amounts of history. GA4 availability is also limited in this slice, so the
initial feature frame deliberately uses GSC measurements rather than
assuming unavailable analytics values are zero.

The `decline_proxy` is a defined threshold-based outcome: average daily
impressions in March 16–31 are compared with March 1–15. It is not proof of
a sustained future decline, and it does not show that a content refresh
would cause recovery.

The baseline Precision@50 of 0.400 is an in-slice benchmark, not an
estimate of performance on unseen clients. A stronger evaluation would
use client-grouped validation and test whether the ranking generalizes.

The deliberate leakage experiment produced Precision@50 of 1.000 because
the score copied the target. That result is invalid and must not be reported
as model performance. The retained benchmark is the honest baseline,
Precision@50 = 0.400.

In [ ]:
# Summarize the limits and retained benchmark without scanning the warehouse again.

print("DATA LIMITS — CURRENT DEVELOPMENT SLICE")
print(f"Client-content rows: {len(feature_frame):,}")
print(
    "Distinct pseudonymized clients:",
    feature_frame["client_hash_id"].nunique()
)
print(f"Proxy-positive rate: {feature_frame['decline_proxy'].mean():.1%}")

print("\nMissing feature values:")
display(
    feature_frame[FEATURE_COLUMNS]
    .isna()
    .sum()
    .rename("missing_values")
    .to_frame()
)

print(f"\nHonest baseline Precision@50: {honest_precision_50:.3f}")
print("Leaked score retained as a feature: False")

DATA LIMITS — CURRENT DEVELOPMENT SLICE
Client-content rows: 96,676
Distinct pseudonymized clients: 37
Proxy-positive rate: 31.2%

Missing feature values:


,missing_values
impressions_per_day_early,0
clicks_per_day_early,0
ctr_early,0
avg_position_early,6
active_days_early,0



Honest baseline Precision@50: 0.420
Leaked score retained as a feature: False


## Self-check

- [x] Every section is filled — contract reasoning and supporting code are present.
- [x] The notebook runs top to bottom with no errors (Runtime → Run all).
- [x] No client names, raw URLs, or private queries have been included.
- [x] Claims use careful wording: observed, measured, directional, decision-support.
- [x] The completed notebook is committed to `work/notebooks/` in my repository.